In [2]:
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("../final_training_data")

FINAL_CSV = OUTPUT_DIR / "final_resume_skill_ner_dataset.csv"

training_dataset = pd.read_csv(FINAL_CSV)

print("Dataset loaded successfully!")
print("Shape:", training_dataset.shape)
print("\nColumns:")
print(training_dataset.columns.tolist())

Dataset loaded successfully!
Shape: (9998, 7)

Columns:
['resume_id', 'resume_text', 'tokens', 'ner_tags', 'skills', 'num_skills', 'ner_ids']


In [3]:
import ast

list_columns = [
    "tokens",
    "ner_tags",
    "skills",
    "ner_ids"
]

for col in list_columns:
    if col in training_dataset.columns:
        training_dataset[col] = training_dataset[col].apply(
            lambda x: ast.literal_eval(x)
            if isinstance(x, str)
            else x
        )

print("List columns converted successfully.")

List columns converted successfully.


In [4]:
print(training_dataset.iloc[0]["tokens"])
print()
print(training_dataset.iloc[0]["ner_tags"])
print()
print(training_dataset.iloc[0]["ner_ids"])

['Education', ':', 'Bachelor', "'", 's', 'in', 'Computer', 'Science', 'Experience', ':', '2', 'years', 'Skills', ':', 'Market', 'Research', ',', 'Maven', ',', 'Java', ',', 'REST', 'API', ',', 'Spring', 'Boot', ',', 'Microservices', ',', 'Statistics', ',', 'SQL']

['O', 'O', 'O', 'O', 'O', 'O', 'B-SKILL', 'I-SKILL', 'O', 'O', 'O', 'O', 'O', 'O', 'B-SKILL', 'I-SKILL', 'O', 'B-SKILL', 'O', 'B-SKILL', 'O', 'B-SKILL', 'I-SKILL', 'O', 'O', 'O', 'O', 'B-SKILL', 'O', 'B-SKILL', 'O', 'B-SKILL']

[0, 0, 0, 0, 0, 0, 1, 2, 0, 0, 0, 0, 0, 0, 1, 2, 0, 1, 0, 1, 0, 1, 2, 0, 0, 0, 0, 1, 0, 1, 0, 1]


In [5]:
from sklearn.model_selection import train_test_split

# 80% training, 20% temporary
train_df, temp_df = train_test_split(
    training_dataset,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

# 10% validation, 10% test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train      :", train_df.shape)
print("Validation :", val_df.shape)
print("Test       :", test_df.shape)

Train      : (7998, 7)
Validation : (1000, 7)
Test       : (1000, 7)


In [6]:
train_df.to_csv(
    OUTPUT_DIR / "train.csv",
    index=False
)

val_df.to_csv(
    OUTPUT_DIR / "validation.csv",
    index=False
)

test_df.to_csv(
    OUTPUT_DIR / "test.csv",
    index=False
)

print("All split datasets saved successfully!")

All split datasets saved successfully!


In [1]:
import pandas as pd
import numpy as np
import ast
import json
import os

from pathlib import Path

import torch

from datasets import Dataset, DatasetDict

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer
)

import evaluate

In [2]:
print("PyTorch version:", torch.__version__)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Training will run on CPU.")

PyTorch version: 2.13.0+cu126
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 A Laptop GPU


In [3]:
import pandas as pd
import ast

train_df = pd.read_csv("../final_training_data/train.csv")
val_df = pd.read_csv("../final_training_data/validation.csv")
test_df = pd.read_csv("../final_training_data/test.csv")

# Convert list-like columns back to Python lists
for df in [train_df, val_df, test_df]:
    df["tokens"] = df["tokens"].apply(ast.literal_eval)
    df["ner_tags"] = df["ner_tags"].apply(ast.literal_eval)
    df["ner_ids"] = df["ner_ids"].apply(ast.literal_eval)
    df["skills"] = df["skills"].apply(ast.literal_eval)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

train_df.head()

Train: (7998, 7)
Validation: (1000, 7)
Test: (1000, 7)


,resume_id,resume_text,tokens,ner_tags,skills,num_skills,ner_ids
0,R008965,Education: Bachelor's in Accounting Experience...,"[Education, :, Bachelor, ', s, in, Accounting,...","[O, O, O, O, O, O, B-SKILL, O, O, O, O, O, O, ...","[accounting, risk assessment, leadership, php,...",5,"[0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,R001562,Education: License Experience: 1 years Skills:...,"[Education, :, License, Experience, :, 1, year...","[O, O, O, O, O, O, O, O, O, O, O, B-SKILL, I-S...","[customer service, flutter, kubernetes, adapta...",5,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 0, 1, ..."
2,R001671,Education: High School Experience: 0 years Ski...,"[Education, :, High, School, Experience, :, 0,...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[jupyter, documentation, pandas]",3,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,R004346,Education: Enterprise Architecture Certificati...,"[Education, :, Enterprise, Architecture, Certi...","[O, O, O, O, O, O, O, O, O, O, O, B-SKILL, I-S...","[technical writing, leadership, decision makin...",4,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 0, 1, ..."
4,R008100,Education: Bachelor's in Supply Chain Manageme...,"[Education, :, Bachelor, ', s, in, Supply, Cha...","[O, O, O, O, O, O, B-SKILL, I-SKILL, I-SKILL, ...","[supply chain management, go]",2,"[0, 0, 0, 0, 0, 0, 1, 2, 2, 0, 0, 0, 0, 0, 0, ..."


In [4]:
from datasets import Dataset, DatasetDict

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

dataset = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset
})

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['resume_id', 'resume_text', 'tokens', 'ner_tags', 'skills', 'num_skills', 'ner_ids'],
        num_rows: 7998
    })
    validation: Dataset({
        features: ['resume_id', 'resume_text', 'tokens', 'ner_tags', 'skills', 'num_skills', 'ner_ids'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['resume_id', 'resume_text', 'tokens', 'ner_tags', 'skills', 'num_skills', 'ner_ids'],
        num_rows: 1000
    })
})


In [5]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded successfully!")
print("Vocabulary size:", tokenizer.vocab_size)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

c:\Users\pushk\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\pushk\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizer loaded successfully!
Vocabulary size: 30522


In [6]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        max_length=512,
        is_split_into_words=True
    )

    all_labels = []

    for i, labels in enumerate(examples["ner_ids"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)

        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:

            # Special tokens: [CLS], [SEP], [PAD]
            if word_idx is None:
                label_ids.append(-100)

            # First subword of a word
            elif word_idx != previous_word_idx:
                label_ids.append(labels[word_idx])

            # Additional subwords
            else:
                original_label = labels[word_idx]

                # B-SKILL → I-SKILL for subsequent subwords
                if original_label == 1:
                    label_ids.append(2)
                else:
                    label_ids.append(original_label)

            previous_word_idx = word_idx

        all_labels.append(label_ids)

    tokenized_inputs["labels"] = all_labels

    return tokenized_inputs


print("Label alignment function created successfully!")

Label alignment function created successfully!


In [7]:
tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names
)

print(tokenized_dataset)

Map:   0%|          | 0/7998 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 7998
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 1000
    })
})


In [9]:
# Define label mappings

label2id = {
    "O": 0,
    "B-SKILL": 1,
    "I-SKILL": 2
}

id2label = {
    0: "O",
    1: "B-SKILL",
    2: "I-SKILL"
}

print("Label mapping created successfully!")

print(id2label)

Label mapping created successfully!
{0: 'O', 1: 'B-SKILL', 2: 'I-SKILL'}


In [10]:
sample = tokenized_dataset["train"][0]

print("Tokens:")
print(tokenizer.convert_ids_to_tokens(sample["input_ids"]))

print("\nLabels:")
print(sample["labels"])

print("\nNumber of input tokens:", len(sample["input_ids"]))
print("Number of labels:", len(sample["labels"]))

print("\nLabel mapping:")
print(id2label)

Tokens:
['[CLS]', 'education', ':', 'bachelor', "'", 's', 'in', 'accounting', 'experience', ':', '5', 'years', 'skills', ':', 'communication', ',', 'r', ',', 'risk', 'assessment', ',', 'ci', '/', 'cd', ',', 'leadership', ',', 'php', ',', 'internal', 'controls', ',', 'compliance', '[SEP]']

Labels:
[-100, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, -100]

Number of input tokens: 34
Number of labels: 34

Label mapping:
{0: 'O', 1: 'B-SKILL', 2: 'I-SKILL'}


In [11]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

print("DistilBERT NER model created successfully!")
print("Number of labels:", model.config.num_labels)
print("Labels:", model.config.id2label)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBERT NER model created successfully!
Number of labels: 3
Labels: {0: 'O', 1: 'B-SKILL', 2: 'I-SKILL'}


In [12]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

print("Data collator created successfully!")

Data collator created successfully!


In [13]:
import evaluate
import numpy as np

seqeval = evaluate.load("seqeval")

print("Seqeval loaded successfully!")

Seqeval loaded successfully!


In [14]:
def compute_metrics(eval_prediction):
    predictions, labels = eval_prediction

    # Convert model outputs to predicted label IDs
    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []

    for prediction, label in zip(predictions, labels):
        pred_list = []
        label_list = []

        for pred, lab in zip(prediction, label):

            # Ignore special tokens and padding
            if lab == -100:
                continue

            pred_list.append(id2label[pred])
            label_list.append(id2label[lab])

        true_predictions.append(pred_list)
        true_labels.append(label_list)

    results = seqeval.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"]
    }


print("Metrics function created successfully!")

Metrics function created successfully!


In [15]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./skill_ner_model",

    # Training
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,

    # Batch sizes
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    # Evaluation & saving
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,

    # Best model
    metric_for_best_model="f1",
    greater_is_better=True,

    # Logging
    logging_steps=100,
    report_to="none",

    # Performance
    fp16=True
)

print("Training configuration created successfully!")

Training configuration created successfully!


In [16]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],

    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer created successfully!")

Trainer created successfully!


In [17]:
# Check dataset sizes
print("Training samples:", len(tokenized_dataset["train"]))
print("Validation samples:", len(tokenized_dataset["validation"]))
print("Test samples:", len(tokenized_dataset["test"]))

# Check model device
print("Model device:", model.device)

# Check label mapping
print("Labels:", model.config.id2label)
print("Number of labels:", model.config.num_labels)

Training samples: 7998
Validation samples: 1000
Test samples: 1000
Model device: cuda:0
Labels: {0: 'O', 1: 'B-SKILL', 2: 'I-SKILL'}
Number of labels: 3


In [18]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.004377,0.000496,0.999394,0.999798,0.999596,0.999882
2,0.000665,0.000064,1.000000,1.000000,1.000000,1.000000
3,0.000130,0.000045,1.000000,1.000000,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.019040214898064733, metrics={'train_runtime': 382.2463, 'train_samples_per_second': 62.771, 'train_steps_per_second': 7.848, 'total_flos': 249398821314000.0, 'train_loss': 0.019040214898064733, 'epoch': 3.0})

In [19]:
test_results = trainer.evaluate(
    tokenized_dataset["test"]
)

print("Test Results:")
for key, value in test_results.items():
    print(f"{key}: {value}")

Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy
0.000130,0.000064,3,1.000000,1.000000,1.000000,1.000000


Test Results:
eval_loss: 6.4023734012153e-05
eval_precision: 1.0
eval_recall: 1.0
eval_f1: 1.0
eval_accuracy: 1.0


In [23]:
from pathlib import Path

# Root-level ml_models folder
model_save_path = Path("../../../ml_models/skill_ner_model")

# Create folder
model_save_path.mkdir(parents=True, exist_ok=True)

# Save trained model
trainer.save_model(str(model_save_path))

# Save tokenizer
tokenizer.save_pretrained(str(model_save_path))

print("Model saved successfully!")
print("Location:", model_save_path.resolve())

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully!
Location: C:\Users\pushk\full_stack_projects\EDI-final\ml_models\skill_ner_model


In [21]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

def extract_skills(resume_text):
    """
    Extract skills from a resume using the trained DistilBERT NER model.
    """

    # Basic tokenization
    words = resume_text.split()

    # Tokenize while keeping word boundaries
    inputs = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    inputs = {key: value.to(device) for key, value in inputs.items()}

    # Model prediction
    with torch.no_grad():
        outputs = model(**inputs)

    predictions = torch.argmax(outputs.logits, dim=2)[0].cpu().numpy()

    word_ids = inputs["input_ids"][0].cpu().numpy()

    # Get word mapping from tokenizer
    tokenized = tokenizer(
        words,
        is_split_into_words=True,
        truncation=True,
        max_length=512
    )

    word_mapping = tokenized.word_ids()

    extracted_skills = []
    current_skill = []

    previous_word = None

    for pred, word_idx in zip(predictions, word_mapping):

        if word_idx is None:
            continue

        label = id2label[pred]

        # Avoid processing the same word multiple times
        if word_idx == previous_word:
            continue

        word = words[word_idx]

        if label == "B-SKILL":

            if current_skill:
                extracted_skills.append(" ".join(current_skill))

            current_skill = [word]

        elif label == "I-SKILL":

            if current_skill:
                current_skill.append(word)

        else:

            if current_skill:
                extracted_skills.append(" ".join(current_skill))
                current_skill = []

        previous_word = word_idx

    # Add final skill
    if current_skill:
        extracted_skills.append(" ".join(current_skill))

    return extracted_skills


print("Skill extraction function created successfully!")

Skill extraction function created successfully!


In [24]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

MODEL_PATH = "../../../ml_models/skill_ner_model"

test_tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
test_model = AutoModelForTokenClassification.from_pretrained(MODEL_PATH)

print("Saved model loaded successfully!")
print("Labels:", test_model.config.id2label)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Saved model loaded successfully!
Labels: {0: 'O', 1: 'B-SKILL', 2: 'I-SKILL'}


In [25]:
import torch

# Use saved model for inference
inference_model = test_model
inference_tokenizer = test_tokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inference_model.to(device)
inference_model.eval()


def extract_skills(resume_text):
    words = resume_text.split()

    inputs = inference_tokenizer(
        words,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    word_ids = inputs.word_ids(batch_index=0)

    model_inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = inference_model(**model_inputs)

    predictions = torch.argmax(outputs.logits, dim=-1)[0].cpu().numpy()

    extracted_skills = []
    current_skill = []
    previous_word_idx = None

    for prediction, word_idx in zip(predictions, word_ids):

        if word_idx is None:
            continue

        # Skip repeated subwords
        if word_idx == previous_word_idx:
            continue

        word = words[word_idx]
        label = inference_model.config.id2label[int(prediction)]

        if label == "B-SKILL":

            if current_skill:
                extracted_skills.append(" ".join(current_skill))

            current_skill = [word]

        elif label == "I-SKILL":

            if current_skill:
                current_skill.append(word)

        else:

            if current_skill:
                extracted_skills.append(" ".join(current_skill))
                current_skill = []

        previous_word_idx = word_idx

    if current_skill:
        extracted_skills.append(" ".join(current_skill))

    return extracted_skills


print("Skill extraction pipeline ready!")

Skill extraction pipeline ready!


In [26]:
sample_resume = """
I am a Full Stack Developer with experience in Python,
JavaScript, React, Node.js, Express.js, MongoDB, Docker,
Git, REST API development, and Machine Learning.
"""

skills = extract_skills(sample_resume)

print("Extracted Skills:")
for skill in skills:
    print("-", skill)

Extracted Skills:
- Python,
- JavaScript,
- React,
- MongoDB,
- Docker,
- Git,
- REST API
- Machine Learning.


In [27]:
import re

def clean_skills(skills):
    cleaned = []

    for skill in skills:
        # Remove punctuation from beginning/end
        skill = skill.strip(" ,.;:!?()[]{}")

        # Remove extra spaces
        skill = re.sub(r"\s+", " ", skill).strip()

        if skill and skill.lower() not in [s.lower() for s in cleaned]:
            cleaned.append(skill)

    return cleaned


cleaned_skills = clean_skills(skills)

print("Cleaned Extracted Skills:")
for skill in cleaned_skills:
    print("-", skill)

Cleaned Extracted Skills:
- Python
- JavaScript
- React
- MongoDB
- Docker
- Git
- REST API
- Machine Learning


In [28]:
import pandas as pd

VOCAB_PATH = "../final_training_data/master_skill_vocabulary.csv"

master_vocab_df = pd.read_csv(VOCAB_PATH)

print("Vocabulary shape:", master_vocab_df.shape)
print("Columns:")
print(master_vocab_df.columns.tolist())

master_vocab_df.head()

Vocabulary shape: (99886, 3)
Columns:
['canonical_skill', 'normalized_skill', 'word_count']


,canonical_skill,normalized_skill,word_count
0,create pipeline designs accounting for the coa...,create pipeline designs accounting for the coa...,17
1,ability to comprehend spoken and written Ancie...,ability to comprehend spoken and written ancie...,16
2,ensure navigation equipment is prepared for us...,ensure navigation equipment is prepared for us...,16
3,ability to comprehend spoken and written Limbu...,ability to comprehend spoken and written limbu...,15
4,discuss with social services stakeholders disc...,discuss with social services stakeholders disc...,15


In [29]:
# Create a clean skill vocabulary

vocab = master_vocab_df.copy()

# Remove missing values
vocab = vocab.dropna(subset=["canonical_skill", "normalized_skill"])

# Remove duplicates
vocab = vocab.drop_duplicates(subset=["normalized_skill"])

# Keep reasonable skill phrase lengths
vocab = vocab[
    (vocab["word_count"] >= 1) &
    (vocab["word_count"] <= 6)
].copy()

# Create lookup dictionary
skill_lookup = dict(
    zip(
        vocab["normalized_skill"].str.lower().str.strip(),
        vocab["canonical_skill"].str.strip()
    )
)

print("Original vocabulary:", len(master_vocab_df))
print("Clean vocabulary:", len(vocab))
print("Lookup entries:", len(skill_lookup))

print("\nSample skills:")
for skill in list(skill_lookup.values())[:20]:
    print("-", skill)

Original vocabulary: 99886
Clean vocabulary: 91254
Lookup entries: 91254

Sample skills:
- 3D and 2D CAD for footwear
- a cultural custom of food preparation
- a manufacturer's instructions for audio-visual equipment
- a metal and metal ore product
- a prepared animal feeds contamination hazard
- a type of hand-operated printing devices
- abide by regulations on banned materials
- abide by safety precautions in printing
- abide to airport unpaved areas ordinances
- able to use online communication tools
- accept accountability in inland water transport
- accept accountability in inland water transportation
- accept business ethical code of conducts
- accommodate cargo in cargo transport vehicle
- accommodate cargo in freight transport vehicle
- accommodate freight in cargo transport vehicle
- accommodate freight in freight transport vehicle
- accompany students on a field trip
- accomplish physical stress tests with models
- accomplish transaction procedures for returned vehicles


In [30]:
# Analyze vocabulary size by number of words

word_count_stats = (
    vocab["word_count"]
    .value_counts()
    .sort_index()
)

print("Vocabulary by word count:")
print(word_count_stats)

print("\nTotal entries:", len(vocab))

Vocabulary by word count:
word_count
1     1751
2    13654
3    30032
4    23028
5    14326
6     8463
Name: count, dtype: int64

Total entries: 91254


In [31]:
# Inspect vocabulary entries that look like actions/activities

action_words = [
    "accept", "accommodate", "accompany", "accomplish",
    "act", "adapt", "administer", "advise", "analyse",
    "analyze", "apply", "assess", "assist", "build",
    "calculate", "carry", "check", "collect", "communicate",
    "conduct", "coordinate", "create", "develop", "ensure",
    "evaluate", "handle", "implement", "manage", "monitor",
    "operate", "organize", "perform", "prepare", "provide",
    "repair", "research", "review", "support", "test",
    "train", "use", "work"
]

action_pattern = r"^(?:" + "|".join(action_words) + r")\b"

action_vocab = vocab[
    vocab["normalized_skill"]
    .str.lower()
    .str.strip()
    .str.contains(action_pattern, regex=True, na=False)
].copy()

print("Action/activity entries:", len(action_vocab))

print("\nSample action/activity entries:")
for skill in action_vocab["canonical_skill"].head(30):
    print("-", skill)

Action/activity entries: 21001

Sample action/activity entries:
- accept accountability in inland water transport
- accept accountability in inland water transportation
- accept business ethical code of conducts
- accommodate cargo in cargo transport vehicle
- accommodate cargo in freight transport vehicle
- accommodate freight in cargo transport vehicle
- accommodate freight in freight transport vehicle
- accompany students on a field trip
- accomplish physical stress tests with models
- accomplish transaction procedures for returned vehicles
- act appropriately to changing operational requirements
- act as an advocate for homeopathy
- act as an expert dance consultant
- act as an expert witness psychologist
- act as intermediary in digital technology
- act as leader in dental teams
- act as leader of dental team
- act as leader of dental teams
- act ethically in providing transport services
- act in accordance with signalling instructions
- act in accordance with traffic signals
- ac

In [32]:
# Create a more resume-focused vocabulary
# Keep short skill phrases and remove obvious activity/action phrases

skill_vocab = vocab.copy()

# Remove entries that start with common activity/action verbs
skill_vocab = skill_vocab[
    ~skill_vocab["normalized_skill"].str.lower().str.strip().str.startswith(
        tuple(action_words)
    )
].copy()

# Remove very long phrases
skill_vocab = skill_vocab[
    skill_vocab["word_count"] <= 4
].copy()

# Remove duplicates
skill_vocab = skill_vocab.drop_duplicates(
    subset=["normalized_skill"]
)

print("Original clean vocabulary:", len(vocab))
print("Resume-focused vocabulary:", len(skill_vocab))

print("\nSample entries:")
for skill in skill_vocab["canonical_skill"].head(30):
    print("-", skill)

Original clean vocabulary: 91254
Resume-focused vocabulary: 52900

Sample entries:
- 2D CAD for footwear
- 3D body scan technologies
- 3D body scanning technologies
- 3D CAD for footwear
- 3D MMORPG Game Engine
- 3D Surface Texturing Techniques
- 5S workplace organisation method
- 5S workplace organization method
- a brand marketing technique
- a business management principle
- a combination of flavours
- a combination of textures
- a contents marketing strategy
- a continuous improvement philosophy
- a credit card payments
- a credit control process
- a data mining method
- a debt collection technique
- a floral composition technique
- a food safety principle
- a gem cutting form
- a grain dehydration recipe
- a grain-for-beverage milling process
- a honey extractors type
- a manufacturer's recommended price
- a modern brewing system
- a musical instrument accessory
- a paint spraying technique
- a plant care product
- a product coding system


In [36]:
import re
import torch

# ---------------------------------------------------------
# Normalize text for matching
# ---------------------------------------------------------
def normalize_skill_text(text):
    text = str(text).lower().strip()

    # Normalize common punctuation
    text = text.replace("–", "-")
    text = text.replace("—", "-")
    
    # Convert punctuation separators to spaces
    text = re.sub(r"[._/]+", " ", text)
    text = re.sub(r"[-]+", " ", text)

    # Remove remaining special characters
    text = re.sub(r"[^a-z0-9+# ]", " ", text)

    # Normalize spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


# ---------------------------------------------------------
# Prepare vocabulary lookup
# ---------------------------------------------------------
normalized_vocab = {}

for _, row in skill_vocab.iterrows():
    key = normalize_skill_text(row["normalized_skill"])
    canonical = str(row["canonical_skill"]).strip()

    if key:
        normalized_vocab[key] = canonical


# Add aliases
for alias, canonical in skill_aliases.items():
    normalized_vocab[normalize_skill_text(alias)] = canonical


print("Total vocabulary + aliases:", len(normalized_vocab))


# ---------------------------------------------------------
# Vocabulary phrase matching
# ---------------------------------------------------------
def vocabulary_match(resume_text):
    text = normalize_skill_text(resume_text)
    words = text.split()

    matches = []

    # Longest phrases first
    max_words = 4

    for n in range(max_words, 0, -1):
        for i in range(len(words) - n + 1):

            phrase = " ".join(words[i:i+n])

            if phrase in normalized_vocab:
                matches.append(normalized_vocab[phrase])

    # Remove duplicates
    unique_matches = []
    seen = set()

    for skill in matches:
        key = skill.lower()

        if key not in seen:
            seen.add(key)
            unique_matches.append(skill)

    return unique_matches


# ---------------------------------------------------------
# Test vocabulary matching
# ---------------------------------------------------------
test_resume = """
I am a Full Stack Developer with experience in Python,
JavaScript, React, Node.js, Express.js, MongoDB, Docker,
Git, REST API development, and Machine Learning.
"""

matched_skills = vocabulary_match(test_resume)

print("\nVocabulary matches:")
for skill in matched_skills:
    print("✅", skill)

Total vocabulary + aliases: 52816

Vocabulary matches:
✅ Node.js
✅ Express.js
✅ REST API
✅ Machine Learning
✅ Python
✅ JavaScript
✅ React
✅ MongoDB
✅ Docker
✅ git


In [37]:
def hybrid_extract_skills(resume_text):
    # -----------------------------------------
    # 1. NER model extraction
    # -----------------------------------------
    ner_skills = extract_skills(resume_text)

    # -----------------------------------------
    # 2. Clean NER results
    # -----------------------------------------
    ner_skills = clean_skills(ner_skills)

    # -----------------------------------------
    # 3. Vocabulary matching
    # -----------------------------------------
    vocab_skills = vocabulary_match(resume_text)

    # -----------------------------------------
    # 4. Combine both
    # -----------------------------------------
    combined = ner_skills + vocab_skills

    # -----------------------------------------
    # 5. Deduplicate
    # -----------------------------------------
    final_skills = []
    seen = set()

    for skill in combined:
        skill = str(skill).strip()

        if not skill:
            continue

        key = normalize_skill_text(skill)

        if key not in seen:
            seen.add(key)
            final_skills.append(skill)

    return final_skills


# ---------------------------------------------------------
# Test hybrid extractor
# ---------------------------------------------------------

test_resume = """
I am a Full Stack Developer with experience in Python,
JavaScript, React, Node.js, Express.js, MongoDB, Docker,
Git, REST API development, and Machine Learning.
"""

final_skills = hybrid_extract_skills(test_resume)

print("Final extracted skills:\n")

for skill in final_skills:
    print("✅", skill)

print("\nTotal skills:", len(final_skills))

Final extracted skills:

✅ Python
✅ JavaScript
✅ React
✅ MongoDB
✅ Docker
✅ Git
✅ REST API
✅ Machine Learning
✅ Node.js
✅ Express.js

Total skills: 10


In [39]:
import pandas as pd
import ast

DATASET_PATH = "../final_training_data/final_resume_skill_ner_dataset.csv"

final_dataset = pd.read_csv(DATASET_PATH)

print("Dataset shape:", final_dataset.shape)
print("Columns:")
print(final_dataset.columns.tolist())

Dataset shape: (9998, 7)
Columns:
['resume_id', 'resume_text', 'tokens', 'ner_tags', 'skills', 'num_skills', 'ner_ids']


In [40]:
# Test hybrid extraction on 10 real resumes

for i in range(10):
    resume = final_dataset.iloc[i]["resume_text"]

    skills = hybrid_extract_skills(resume)

    print("=" * 80)
    print(f"Resume {i + 1}")
    print("Skills:", skills)
    print("Total:", len(skills))

Resume 1
Skills: ['Computer Science', 'Market Research', 'Maven', 'Java', 'REST API', 'Microservices', 'Statistics', 'SQL']
Total: 8
Resume 2
Skills: ['Microbiology', 'Agile', 'Data Analysis', 'Scala', 'Business Development']
Total: 5
Resume 3
Skills: ['React', 'Customer Service', 'PostgreSQL']
Total: 3
Resume 4
Skills: ['Computer Science', 'MySQL', 'Query Optimization', 'Terraform', 'PHP', 'HTML/CSS', 'Financial Analysis', 'HTML', 'CSS']
Total: 9
Resume 5
Skills: ['Design', 'Docker', 'Scrum', 'Ansible', 'Leadership', 'Design Principles']
Total: 6
Resume 6
Skills: ['Database Design', 'PyTorch', 'Deep Learning', 'CI/CD', 'database', 'design']
Total: 6
Resume 7
Skills: ['Computer Science', 'Cassandra', 'NoSQL', 'Business Development', 'Machine Learning']
Total: 5
Resume 8
Skills: ['IT', 'SQL', 'ETL', 'JavaScript', 'Kanban', 'Web Accessibility', 'Scikit-learn']
Total: 7
Resume 9
Skills: ['Swift', 'Physical Therapy', 'PHP']
Total: 3
Resume 10
Skills: ['Computer Science', 'PostgreSQL', 'Kan

In [41]:
def remove_redundant_skills(skills):
    # Clean
    skills = [
        str(skill).strip()
        for skill in skills
        if str(skill).strip()
    ]

    # Remove exact duplicates
    unique = []
    seen = set()

    for skill in skills:
        key = normalize_skill_text(skill)

        if key not in seen:
            seen.add(key)
            unique.append(skill)

    # Sort longest phrases first
    unique_sorted = sorted(
        unique,
        key=lambda x: len(normalize_skill_text(x).split()),
        reverse=True
    )

    final = []

    for skill in unique_sorted:
        skill_norm = normalize_skill_text(skill)

        # Check if this skill is already completely covered
        # by a longer extracted skill
        redundant = False

        for existing in final:
            existing_norm = normalize_skill_text(existing)

            if (
                skill_norm != existing_norm
                and f" {skill_norm} " in f" {existing_norm} "
            ):
                redundant = True
                break

        if not redundant:
            final.append(skill)

    # Restore original-style ordering
    return final


# Test on Resume 6 example
test_skills = [
    "Database Design",
    "PyTorch",
    "Deep Learning",
    "CI/CD",
    "database",
    "design"
]

cleaned = remove_redundant_skills(test_skills)

print("Before:")
print(test_skills)

print("\nAfter:")
for skill in cleaned:
    print("✅", skill)

Before:
['Database Design', 'PyTorch', 'Deep Learning', 'CI/CD', 'database', 'design']

After:
✅ Database Design
✅ Deep Learning
✅ CI/CD
✅ PyTorch


In [42]:
def hybrid_extract_skills(resume_text):
    # 1. NER model
    ner_skills = extract_skills(resume_text)

    # 2. Clean NER output
    ner_skills = clean_skills(ner_skills)

    # 3. Vocabulary matching
    vocab_skills = vocabulary_match(resume_text)

    # 4. Combine
    combined = ner_skills + vocab_skills

    # 5. Remove duplicates and redundant sub-skills
    final_skills = remove_redundant_skills(combined)

    return final_skills


# ---------------------------------------------------------
# Test
# ---------------------------------------------------------

test_resume = """
I am a Full Stack Developer with experience in Python,
JavaScript, React, Node.js, Express.js, MongoDB, Docker,
Git, REST API development, and Machine Learning.
"""

final_skills = hybrid_extract_skills(test_resume)

print("Final extracted skills:\n")

for skill in final_skills:
    print("✅", skill)

print("\nTotal skills:", len(final_skills))

Final extracted skills:

✅ REST API
✅ Machine Learning
✅ Node.js
✅ Express.js
✅ Python
✅ JavaScript
✅ React
✅ MongoDB
✅ Docker
✅ Git

Total skills: 10


In [43]:
# Test hybrid extractor on 100 real resumes

results = []

for i in range(min(100, len(final_dataset))):
    resume = final_dataset.iloc[i]["resume_text"]

    skills = hybrid_extract_skills(resume)

    results.append({
        "resume_id": final_dataset.iloc[i]["resume_id"],
        "skills": skills,
        "num_skills": len(skills)
    })

results_df = pd.DataFrame(results)

print("Resumes processed:", len(results_df))
print("Average skills per resume:", round(results_df["num_skills"].mean(), 2))
print("Minimum skills:", results_df["num_skills"].min())
print("Maximum skills:", results_df["num_skills"].max())

print("\nFirst 10 results:")
display(results_df.head(10))

Resumes processed: 100
Average skills per resume: 5.01
Minimum skills: 2
Maximum skills: 9

First 10 results:


,resume_id,skills,num_skills
0,R000000,"[Computer Science, Market Research, REST API, ...",8
1,R000001,"[Data Analysis, Business Development, Microbio...",5
2,R000002,"[Customer Service, React, PostgreSQL]",3
3,R000003,"[Computer Science, Query Optimization, HTML/CS...",7
4,R000004,"[Design Principles, Docker, Scrum, Ansible, Le...",5
5,R000005,"[Database Design, Deep Learning, CI/CD, PyTorch]",4
6,R000006,"[Computer Science, Business Development, Machi...",5
7,R000007,"[Web Accessibility, Scikit-learn, IT, SQL, ETL...",7
8,R000008,"[Physical Therapy, Swift, PHP]",3
9,R000009,"[Computer Science, PostgreSQL, Kanban, Complia...",5


In [44]:
from collections import Counter

# Count every extracted skill across the 100 resumes
skill_counter = Counter()

for skills in results_df["skills"]:
    for skill in skills:
        skill_counter[skill] += 1

# Convert to DataFrame
skill_frequency_df = pd.DataFrame(
    skill_counter.items(),
    columns=["skill", "resume_count"]
).sort_values(
    "resume_count",
    ascending=False
).reset_index(drop=True)

print("Total unique skills:", len(skill_frequency_df))

print("\nTop 20 extracted skills:")
display(skill_frequency_df.head(20))

Total unique skills: 161

Top 20 extracted skills:


,skill,resume_count
0,Computer Science,19
1,SQL,16
2,Leadership,15
3,MySQL,9
4,Customer Service,9
5,Marketing,9
6,IT,9
7,Adobe Creative Suite,8
8,Documentation,8
9,PHP,6


In [45]:
# Skill category classification

technical_keywords = {
    "python", "java", "javascript", "typescript", "c", "c++", "c#",
    "sql", "mysql", "postgresql", "mongodb", "php", "swift", "kotlin",
    "react", "angular", "vue", "node.js", "express.js",
    "html", "css", "html/css", "docker", "kubernetes",
    "git", "tensorflow", "pytorch", "scikit-learn",
    "machine learning", "deep learning", "rest api",
    "microservices", "terraform", "ansible", "ci/cd",
    "aws", "azure", "google cloud", "linux",
    "data analysis", "statistics", "etl",
    "database design", "query optimization"
}

soft_keywords = {
    "leadership", "communication", "teamwork", "negotiation",
    "customer service", "problem solving", "time management",
    "adaptability", "creativity", "collaboration"
}

tool_keywords = {
    "adobe creative suite", "maven", "jira", "git",
    "docker", "ansible", "terraform", "power bi",
    "tableau", "microsoft excel"
}


def categorize_skill(skill):
    normalized = normalize_skill_text(skill)

    if normalized in technical_keywords:
        return "Technical"

    if normalized in tool_keywords:
        return "Tool"

    if normalized in soft_keywords:
        return "Soft Skill"

    return "Domain / Other"


# Add categories to frequency dataframe
skill_frequency_df["category"] = skill_frequency_df["skill"].apply(
    categorize_skill
)

print("Top extracted skills with categories:")
display(skill_frequency_df.head(20))

Top extracted skills with categories:


,skill,resume_count,category
0,Computer Science,19,Domain / Other
1,SQL,16,Technical
2,Leadership,15,Soft Skill
3,MySQL,9,Technical
4,Customer Service,9,Soft Skill
5,Marketing,9,Domain / Other
6,IT,9,Domain / Other
7,Adobe Creative Suite,8,Tool
8,Documentation,8,Domain / Other
9,PHP,6,Technical


In [46]:
 # Sample Job Description
job_description = """
We are looking for a Full Stack Developer with experience in
Python, JavaScript, React, Node.js, Express.js, MongoDB,
REST API, Docker, Git and Machine Learning.
"""

# Extract required skills from the job description
job_required_skills = hybrid_extract_skills(job_description)

print("Required skills:\n")

for skill in job_required_skills:
    print("✅", skill)

print("\nTotal required skills:", len(job_required_skills))

Required skills:

✅ REST API
✅ Machine Learning
✅ Node.js
✅ Express.js
✅ Python
✅ JavaScript
✅ React
✅ MongoDB
✅ Docker
✅ Git

Total required skills: 10


In [47]:
def compare_resume_with_job(resume_text, job_description):
    # Extract skills
    resume_skills = hybrid_extract_skills(resume_text)
    required_skills = hybrid_extract_skills(job_description)

    # Normalize for comparison
    resume_map = {
        normalize_skill_text(skill): skill
        for skill in resume_skills
    }

    required_map = {
        normalize_skill_text(skill): skill
        for skill in required_skills
    }

    # Find matches
    matched = []
    missing = []

    for key, skill in required_map.items():
        if key in resume_map:
            matched.append(skill)
        else:
            missing.append(skill)

    # ATS skill match percentage
    if len(required_skills) > 0:
        score = (len(matched) / len(required_skills)) * 100
    else:
        score = 0

    return {
        "resume_skills": resume_skills,
        "required_skills": required_skills,
        "matched_skills": matched,
        "missing_skills": missing,
        "score": round(score, 2)
    }


# ---------------------------------------------------------
# Test with a sample resume
# ---------------------------------------------------------

sample_resume = """
I am a Full Stack Developer with experience in Python,
JavaScript, React, Node.js, MongoDB, Docker, Git,
REST API development and Machine Learning.
"""

result = compare_resume_with_job(
    sample_resume,
    job_description
)

print("Matched Skills:")
for skill in result["matched_skills"]:
    print("✅", skill)

print("\nMissing Skills:")
for skill in result["missing_skills"]:
    print("❌", skill)

print("\nATS Skill Match Score:", result["score"], "%")

Matched Skills:
✅ REST API
✅ Machine Learning
✅ Node.js
✅ Python
✅ JavaScript
✅ React
✅ MongoDB
✅ Docker
✅ Git

Missing Skills:
❌ Express.js

ATS Skill Match Score: 90.0 %


In [48]:
def generate_ats_suggestions(result):
    suggestions = []

    # Missing skills
    if result["missing_skills"]:
        for skill in result["missing_skills"]:
            suggestions.append(
                f"Consider adding '{skill}' to your resume if you have relevant experience."
            )

    # General suggestions
    if result["score"] < 50:
        suggestions.append(
            "Your resume has a low skill match with this job description. "
            "Review the required technologies and highlight relevant experience."
        )
    elif result["score"] < 75:
        suggestions.append(
            "Your resume has a moderate skill match. "
            "Add or highlight relevant missing skills where applicable."
        )
    elif result["score"] < 90:
        suggestions.append(
            "Your resume has a strong skill match. "
            "Review the remaining missing skills and relevant project experience."
        )
    else:
        suggestions.append(
            "Your resume has a high skill match with the required skills."
        )

    return suggestions


# Generate suggestions
suggestions = generate_ats_suggestions(result)

print("ATS Suggestions:\n")

for suggestion in suggestions:
    print("💡", suggestion)

ATS Suggestions:

💡 Consider adding 'Express.js' to your resume if you have relevant experience.
💡 Your resume has a high skill match with the required skills.


In [49]:
def analyze_resume_for_job(resume_text, job_description):
    # -----------------------------------------
    # 1. Extract resume skills
    # -----------------------------------------
    resume_skills = hybrid_extract_skills(resume_text)

    # -----------------------------------------
    # 2. Extract job-required skills
    # -----------------------------------------
    required_skills = hybrid_extract_skills(job_description)

    # -----------------------------------------
    # 3. Normalize skills
    # -----------------------------------------
    resume_map = {
        normalize_skill_text(skill): skill
        for skill in resume_skills
    }

    required_map = {
        normalize_skill_text(skill): skill
        for skill in required_skills
    }

    # -----------------------------------------
    # 4. Match skills
    # -----------------------------------------
    matched_skills = []
    missing_skills = []

    for key, skill in required_map.items():
        if key in resume_map:
            matched_skills.append(skill)
        else:
            missing_skills.append(skill)

    # -----------------------------------------
    # 5. Calculate score
    # -----------------------------------------
    if required_skills:
        ats_score = (
            len(matched_skills) / len(required_skills)
        ) * 100
    else:
        ats_score = 0

    # -----------------------------------------
    # 6. Suggestions
    # -----------------------------------------
    suggestions = []

    for skill in missing_skills:
        suggestions.append(
            f"Consider adding '{skill}' to your resume "
            f"if you have relevant experience."
        )

    if ats_score < 50:
        suggestions.append(
            "Review the job requirements and highlight "
            "relevant skills and project experience."
        )
    elif ats_score < 75:
        suggestions.append(
            "Your resume has a moderate skill match. "
            "Highlight relevant experience and missing skills "
            "where applicable."
        )
    elif ats_score < 90:
        suggestions.append(
            "Your resume has a strong skill match. "
            "Review the remaining missing requirements."
        )
    else:
        suggestions.append(
            "Your resume has a high skill match with the "
            "detected job requirements."
        )

    # -----------------------------------------
    # 7. Return complete ATS report
    # -----------------------------------------
    return {
        "ats_score": round(ats_score, 2),
        "resume_skills": resume_skills,
        "required_skills": required_skills,
        "matched_skills": matched_skills,
        "missing_skills": missing_skills,
        "suggestions": suggestions
    }


# ---------------------------------------------------------
# TEST
# ---------------------------------------------------------

ats_report = analyze_resume_for_job(
    sample_resume,
    job_description
)

print("=" * 60)
print("ATS ANALYSIS REPORT")
print("=" * 60)

print("\nATS Score:", ats_report["ats_score"], "%")

print("\nMatched Skills:")
for skill in ats_report["matched_skills"]:
    print("✅", skill)

print("\nMissing Skills:")
for skill in ats_report["missing_skills"]:
    print("❌", skill)

print("\nSuggestions:")
for suggestion in ats_report["suggestions"]:
    print("💡", suggestion)

ATS ANALYSIS REPORT

ATS Score: 90.0 %

Matched Skills:
✅ REST API
✅ Machine Learning
✅ Node.js
✅ Python
✅ JavaScript
✅ React
✅ MongoDB
✅ Docker
✅ Git

Missing Skills:
❌ Express.js

Suggestions:
💡 Consider adding 'Express.js' to your resume if you have relevant experience.
💡 Your resume has a high skill match with the detected job requirements.


In [51]:
def calculate_weighted_ats_score(skill_score, keyword_score):
    # Current weights
    SKILL_WEIGHT = 0.50
    KEYWORD_WEIGHT = 0.20

    weighted_score = (
        skill_score * SKILL_WEIGHT
        + keyword_score * KEYWORD_WEIGHT
    )

    return round(weighted_score, 2)


# Get scores from our previous results
skill_score = ats_report["ats_score"]
keyword_score = keyword_result["keyword_score"]

weighted_score = calculate_weighted_ats_score(
    skill_score,
    keyword_score
)

print("Skill Match Score:", skill_score, "%")
print("Keyword Match Score:", keyword_score, "%")
print("-" * 40)
print("Current Weighted ATS Score:", weighted_score, "%")

Skill Match Score: 90.0 %
Keyword Match Score: 90.0 %
----------------------------------------
Current Weighted ATS Score: 63.0 %


In [52]:
import re


def extract_required_experience(job_description):
    """
    Extract required years of experience from a job description.
    Examples:
    '2 years experience'
    '3+ years of experience'
    '5 years experience'
    """

    patterns = [
        r'(\d+)\s*\+?\s*years?\s+of\s+experience',
        r'(\d+)\s*\+?\s*years?\s+experience',
        r'minimum\s+(\d+)\s*\+?\s*years?',
        r'at\s+least\s+(\d+)\s*\+?\s*years?'
    ]

    text = job_description.lower()

    for pattern in patterns:
        match = re.search(pattern, text)

        if match:
            return int(match.group(1))

    return None


def calculate_experience_match(candidate_experience, job_description):
    required_experience = extract_required_experience(job_description)

    # Job description does not specify experience
    if required_experience is None:
        return {
            "required_experience": None,
            "candidate_experience": candidate_experience,
            "experience_score": 100.0
        }

    # Candidate experience is missing
    if candidate_experience is None:
        return {
            "required_experience": required_experience,
            "candidate_experience": None,
            "experience_score": 0.0
        }

    candidate_experience = float(candidate_experience)

    if candidate_experience >= required_experience:
        score = 100.0
    else:
        score = (
            candidate_experience / required_experience
        ) * 100

    return {
        "required_experience": required_experience,
        "candidate_experience": candidate_experience,
        "experience_score": round(min(score, 100), 2)
    }


# ---------------------------------------------------------
# Test Job Description
# ---------------------------------------------------------

experience_job_description = """
We are looking for a Full Stack Developer with at least
2 years of experience in Python, JavaScript, React,
Node.js, Express.js, MongoDB, Docker and REST API.
"""

# Example candidate with 1 year experience
experience_result = calculate_experience_match(
    candidate_experience=1,
    job_description=experience_job_description
)

print("Required Experience:",
      experience_result["required_experience"], "years")

print("Candidate Experience:",
      experience_result["candidate_experience"], "years")

print("Experience Match Score:",
      experience_result["experience_score"], "%")

Required Experience: 2 years
Candidate Experience: 1.0 years
Experience Match Score: 50.0 %


In [53]:
# ============================================================
# FINAL ATS ANALYZER
# ============================================================

import re
import pandas as pd


# ------------------------------------------------------------
# 1. EDUCATION LEVEL DEFINITIONS
# ------------------------------------------------------------

EDUCATION_LEVELS = {
    "high school": 1,
    "12th": 1,
    "diploma": 2,
    "associate": 2,
    "bachelor": 3,
    "bachelors": 3,
    "b.tech": 3,
    "btech": 3,
    "b.e": 3,
    "be": 3,
    "undergraduate": 3,
    "master": 4,
    "masters": 4,
    "m.tech": 4,
    "mtech": 4,
    "m.e": 4,
    "me": 4,
    "postgraduate": 4,
    "mba": 4,
    "phd": 5,
    "doctorate": 5
}


# ------------------------------------------------------------
# 2. EXTRACT REQUIRED EXPERIENCE FROM JOB DESCRIPTION
# ------------------------------------------------------------

def extract_required_experience(job_description):

    text = job_description.lower()

    patterns = [
        r'(\d+)\s*\+?\s*years?\s+of\s+experience',
        r'(\d+)\s*\+?\s*years?\s+experience',
        r'minimum\s+(\d+)\s*\+?\s*years?',
        r'at\s+least\s+(\d+)\s*\+?\s*years?',
        r'(\d+)\s*-\s*(\d+)\s+years?'
    ]

    for pattern in patterns:

        match = re.search(pattern, text)

        if match:

            # Range such as 2-4 years
            if len(match.groups()) == 2:
                return int(match.group(1))

            return int(match.group(1))

    return None


# ------------------------------------------------------------
# 3. EXPERIENCE MATCH
# ------------------------------------------------------------

def calculate_experience_match(
    candidate_experience,
    job_description
):

    required_experience = extract_required_experience(
        job_description
    )

    # Job doesn't specify experience
    if required_experience is None:

        return {
            "required_experience": None,
            "candidate_experience": candidate_experience,
            "experience_score": 100.0
        }

    # Candidate experience unavailable
    if candidate_experience is None:

        return {
            "required_experience": required_experience,
            "candidate_experience": None,
            "experience_score": 0.0
        }

    candidate_experience = float(candidate_experience)

    # Candidate meets requirement
    if candidate_experience >= required_experience:

        score = 100.0

    # Candidate has partial experience
    else:

        score = (
            candidate_experience /
            required_experience
        ) * 100

    return {
        "required_experience": required_experience,
        "candidate_experience": candidate_experience,
        "experience_score": round(
            min(score, 100),
            2
        )
    }


# ------------------------------------------------------------
# 4. EXTRACT EDUCATION LEVEL
# ------------------------------------------------------------

def extract_education_level(text):

    if not text:
        return None

    text = text.lower()

    # Check highest level first
    education_patterns = [
        ("phd", 5),
        ("doctorate", 5),

        ("master", 4),
        ("masters", 4),
        ("m.tech", 4),
        ("mtech", 4),
        ("m.e", 4),
        ("mba", 4),
        ("postgraduate", 4),

        ("bachelor", 3),
        ("bachelors", 3),
        ("b.tech", 3),
        ("btech", 3),
        ("b.e", 3),
        ("undergraduate", 3),

        ("diploma", 2),
        ("associate", 2),

        ("high school", 1),
        ("12th", 1)
    ]

    for keyword, level in education_patterns:

        if keyword in text:
            return level

    return None


# ------------------------------------------------------------
# 5. REQUIRED EDUCATION FROM JOB DESCRIPTION
# ------------------------------------------------------------

def extract_required_education(job_description):

    return extract_education_level(
        job_description
    )


# ------------------------------------------------------------
# 6. EDUCATION MATCH
# ------------------------------------------------------------

def calculate_education_match(
    candidate_education,
    job_description
):

    required_level = extract_required_education(
        job_description
    )

    # Job doesn't mention education
    if required_level is None:

        return {
            "required_education_level": None,
            "candidate_education_level": candidate_education,
            "education_score": 100.0
        }

    # Candidate education unavailable
    if candidate_education is None:

        return {
            "required_education_level": required_level,
            "candidate_education_level": None,
            "education_score": 0.0
        }

    candidate_level = int(candidate_education)

    if candidate_level >= required_level:

        score = 100.0

    else:

        score = (
            candidate_level /
            required_level
        ) * 100

    return {
        "required_education_level": required_level,
        "candidate_education_level": candidate_level,
        "education_score": round(
            min(score, 100),
            2
        )
    }


# ------------------------------------------------------------
# 7. COMPLETE ATS ANALYZER
# ------------------------------------------------------------

def final_ats_analyzer(
    resume_text,
    job_description,
    candidate_experience=None,
    candidate_education=None
):

    # ========================================================
    # SKILL EXTRACTION
    # ========================================================

    resume_skills = hybrid_extract_skills(
        resume_text
    )

    required_skills = hybrid_extract_skills(
        job_description
    )


    # ========================================================
    # NORMALIZE SKILLS
    # ========================================================

    resume_map = {
        normalize_skill_text(skill): skill
        for skill in resume_skills
    }

    required_map = {
        normalize_skill_text(skill): skill
        for skill in required_skills
    }


    # ========================================================
    # MATCH SKILLS
    # ========================================================

    matched_skills = []
    missing_skills = []

    for key, skill in required_map.items():

        if key in resume_map:

            matched_skills.append(skill)

        else:

            missing_skills.append(skill)


    # ========================================================
    # SKILL MATCH SCORE
    # ========================================================

    if required_skills:

        skill_score = (
            len(matched_skills) /
            len(required_skills)
        ) * 100

    else:

        skill_score = 0


    skill_score = round(
        skill_score,
        2
    )


    # ========================================================
    # KEYWORD MATCH
    # ========================================================

    resume_text_normalized = normalize_skill_text(
        resume_text
    )

    matched_keywords = []
    missing_keywords = []

    for skill in required_skills:

        skill_normalized = normalize_skill_text(
            skill
        )

        if skill_normalized in resume_text_normalized:

            matched_keywords.append(skill)

        else:

            missing_keywords.append(skill)


    if required_skills:

        keyword_score = (
            len(matched_keywords) /
            len(required_skills)
        ) * 100

    else:

        keyword_score = 0


    keyword_score = round(
        keyword_score,
        2
    )


    # ========================================================
    # EXPERIENCE
    # ========================================================

    experience_result = calculate_experience_match(
        candidate_experience,
        job_description
    )

    experience_score = experience_result[
        "experience_score"
    ]


    # ========================================================
    # EDUCATION
    # ========================================================

    education_result = calculate_education_match(
        candidate_education,
        job_description
    )

    education_score = education_result[
        "education_score"
    ]


    # ========================================================
    # FINAL WEIGHTED SCORE
    # ========================================================

    final_score = (

        skill_score * 0.50

        + keyword_score * 0.20

        + experience_score * 0.15

        + education_score * 0.15

    )

    final_score = round(
        final_score,
        2
    )


    # ========================================================
    # SUGGESTIONS
    # ========================================================

    suggestions = []

    # Missing skills
    for skill in missing_skills:

        suggestions.append(
            f"Consider adding '{skill}' "
            f"if you have relevant experience."
        )


    # Experience suggestion
    required_exp = experience_result[
        "required_experience"
    ]

    if required_exp is not None:

        if candidate_experience is None:

            suggestions.append(
                f"Add your years of relevant "
                f"experience to the resume."
            )

        elif candidate_experience < required_exp:

            suggestions.append(
                f"The job description requests "
                f"{required_exp}+ years of experience. "
                f"Your provided experience is "
                f"{candidate_experience} years."
            )


    # Education suggestion
    required_edu = education_result[
        "required_education_level"
    ]

    if required_edu is not None:

        if candidate_education is None:

            suggestions.append(
                "Add your highest educational "
                "qualification to the resume."
            )

        elif candidate_education < required_edu:

            suggestions.append(
                "Review the education requirement "
                "for this position."
            )


    # General suggestion
    if final_score >= 80:

        suggestions.append(
            "The resume has strong alignment "
            "with the analyzed job description."
        )

    elif final_score >= 60:

        suggestions.append(
            "The resume has moderate alignment. "
            "Review the missing skills and "
            "job requirements."
        )

    else:

        suggestions.append(
            "The resume has limited alignment "
            "with the analyzed job description. "
            "Review the required skills, experience "
            "and education."
        )


    # ========================================================
    # RETURN COMPLETE REPORT
    # ========================================================

    return {

        "final_ats_score": final_score,

        "skill_match_score": skill_score,

        "keyword_match_score": keyword_score,

        "experience_match_score": experience_score,

        "education_match_score": education_score,

        "resume_skills": resume_skills,

        "required_skills": required_skills,

        "matched_skills": matched_skills,

        "missing_skills": missing_skills,

        "matched_keywords": matched_keywords,

        "missing_keywords": missing_keywords,

        "required_experience":
            experience_result[
                "required_experience"
            ],

        "candidate_experience":
            experience_result[
                "candidate_experience"
            ],

        "required_education_level":
            education_result[
                "required_education_level"
            ],

        "candidate_education_level":
            education_result[
                "candidate_education_level"
            ],

        "suggestions": suggestions
    }


# ============================================================
# TEST DATA
# ============================================================

sample_resume = """
I am a Full Stack Developer with 1 year of experience
in Python, JavaScript, React, Node.js, MongoDB,
Docker, Git, REST API development and Machine Learning.

I have completed my Bachelor's degree in Computer Engineering.
"""


sample_job_description = """
We are looking for a Full Stack Developer with at least
2 years of experience in Python, JavaScript, React,
Node.js, Express.js, MongoDB, Docker, REST API and
Machine Learning.

Candidates should have a Bachelor's degree in Computer
Science, Computer Engineering or a related field.
"""


# ============================================================
# RUN FINAL ATS ANALYSIS
# ============================================================

ats_report = final_ats_analyzer(

    resume_text=sample_resume,

    job_description=sample_job_description,

    candidate_experience=1,

    candidate_education=3
)


# ============================================================
# DISPLAY COMPLETE REPORT
# ============================================================

print("=" * 70)
print("                 FINAL ATS ANALYSIS REPORT")
print("=" * 70)

print(
    f"\nFINAL ATS SCORE: "
    f"{ats_report['final_ats_score']}%"
)

print("\n" + "-" * 70)
print("SCORE BREAKDOWN")
print("-" * 70)

print(
    f"Skill Match       : "
    f"{ats_report['skill_match_score']}%  × 50%"
)

print(
    f"Keyword Match     : "
    f"{ats_report['keyword_match_score']}%  × 20%"
)

print(
    f"Experience Match  : "
    f"{ats_report['experience_match_score']}%  × 15%"
)

print(
    f"Education Match   : "
    f"{ats_report['education_match_score']}%  × 15%"
)


print("\n" + "-" * 70)
print("REQUIRED SKILLS")
print("-" * 70)

for skill in ats_report["required_skills"]:
    print("•", skill)


print("\n" + "-" * 70)
print("MATCHED SKILLS")
print("-" * 70)

for skill in ats_report["matched_skills"]:
    print("✅", skill)


print("\n" + "-" * 70)
print("MISSING SKILLS")
print("-" * 70)

if ats_report["missing_skills"]:

    for skill in ats_report["missing_skills"]:
        print("❌", skill)

else:

    print("None")


print("\n" + "-" * 70)
print("EXPERIENCE")
print("-" * 70)

print(
    "Required:",
    ats_report["required_experience"],
    "years"
)

print(
    "Candidate:",
    ats_report["candidate_experience"],
    "years"
)


print("\n" + "-" * 70)
print("EDUCATION")
print("-" * 70)

print(
    "Required education level:",
    ats_report["required_education_level"]
)

print(
    "Candidate education level:",
    ats_report["candidate_education_level"]
)


print("\n" + "-" * 70)
print("ATS SUGGESTIONS")
print("-" * 70)

for suggestion in ats_report["suggestions"]:

    print("💡", suggestion)


print("\n" + "=" * 70)
print("                 ANALYSIS COMPLETE")
print("=" * 70)

                 FINAL ATS ANALYSIS REPORT

FINAL ATS SCORE: 79.77%

----------------------------------------------------------------------
SCORE BREAKDOWN
----------------------------------------------------------------------
Skill Match       : 81.82%  × 50%
Keyword Match     : 81.82%  × 20%
Experience Match  : 50.0%  × 15%
Education Match   : 100.0%  × 15%

----------------------------------------------------------------------
REQUIRED SKILLS
----------------------------------------------------------------------
• REST API
• Machine Learning
• Computer Science
• Computer Engineering
• Node.js
• Express.js
• Python
• JavaScript
• React
• MongoDB
• Docker

----------------------------------------------------------------------
MATCHED SKILLS
----------------------------------------------------------------------
✅ REST API
✅ Machine Learning
✅ Computer Engineering
✅ Node.js
✅ Python
✅ JavaScript
✅ React
✅ MongoDB
✅ Docker

----------------------------------------------------------------

In [54]:
# ============================================================
# TEST CURRENT ATS FUNCTION ON YOUR ACTUAL RESUME
# ============================================================

sample_resume = """
Pushkar Chaudhari
Pune, Maharashtra, India
+91-7875231555 | pushkar202006@gmail.com | LinkedIn | GitHub | LeetCode

Profile
Computer Engineering undergraduate passionate about Software Development,
Artificial Intelligence, and Machine Learning. Experienced in developing
full-stack web applications using modern technologies including React.js,
Node.js, Express.js, MongoDB, and cloud deployment platforms. Possess
practical knowledge of machine learning algorithms, data preprocessing,
model training, and evaluation using Python and Scikit-learn.

Education
JSPM Rajarshi Shahu College of Engineering Pune, India
Bachelor of Technology in Computer Engineering Aug 2024 – May 2028

D. S. High School Bhusawal, Maharashtra
Higher Secondary Certificate (Science Stream) Jun 2022 – May 2024

Technical Skills
Languages: JavaScript, Python, Java, C, C++, SQL
Frontend: React.js, HTML5, CSS3, Responsive Web Design
Backend: Node.js, Express.js, REST APIs, JWT Authentication
Databases: MongoDB, Firebase, Supabase
Machine Learning: NumPy, Pandas, Matplotlib, Scikit-learn
Tools: Git, GitHub, Postman, VS Code, MongoDB Compass
Cloud & Deployment: Vercel, Netlify, Render, Google Cloud Run

Projects
AI Test Case Generator | React.js, Node.js, Express.js, MongoDB 2025
Developed a web-based application that generates structured software
test cases from functional requirements using AI-assisted workflows.
Built RESTful backend services for storing, retrieving, editing, and
exporting generated test cases.
Designed responsive user interfaces for requirement submission,
test case review, and project management.
Integrated MongoDB for persistent storage.

Question Paper Management Portal | Node.js, Express.js, Supabase 2025
Developed a secure digital platform for question paper creation,
storage, and distribution.
Implemented JWT-based authentication and role-based authorization.
Designed backend APIs and database workflows using Supabase.

Online Resume Builder | HTML, CSS, JavaScript 2025
Built a client-side resume generation platform with real-time preview.
Implemented dynamic form processing and PDF export features.
Designed responsive templates for desktop and mobile devices.

Certifications
Machine Learning Specialization 2026
DeepLearning.AI & Stanford University (Andrew Ng) Coursera

Backend Web Development 2025
Udemy Online

Achievements
Solved Data Structures and Algorithms problems on LeetCode.
Completed the Machine Learning Specialization by Andrew Ng.
Built and deployed multiple full-stack web applications.
Participated in project-based learning, hackathons, and software
development activities.
"""


# ------------------------------------------------------------
# JOB DESCRIPTION FOR TESTING
# ------------------------------------------------------------

sample_job_description = """
We are looking for a Full Stack Developer with 2 years of experience.

Required skills:
Python, JavaScript, React.js, Node.js, Express.js, MongoDB,
REST API, SQL, Docker, Git, AWS, TypeScript and PostgreSQL.

The candidate should have experience with backend development,
RESTful APIs, database design, cloud deployment and machine learning.

Bachelor's degree in Computer Science, Computer Engineering,
Information Technology or a related field is required.
"""


# ------------------------------------------------------------
# RUN ATS
# ------------------------------------------------------------

actual_resume_ats = final_ats_analyzer(

    resume_text=sample_resume,

    job_description=sample_job_description,

    candidate_experience=0,

    candidate_education=3
)


# ------------------------------------------------------------
# DISPLAY RESULT
# ------------------------------------------------------------

print("=" * 70)
print("             YOUR RESUME - ATS ANALYSIS")
print("=" * 70)

print(
    f"\nFINAL ATS SCORE: "
    f"{actual_resume_ats['final_ats_score']}%"
)


print("\n" + "-" * 70)
print("SCORE BREAKDOWN")
print("-" * 70)

print(
    f"Skill Match       : "
    f"{actual_resume_ats['skill_match_score']}%"
)

print(
    f"Keyword Match     : "
    f"{actual_resume_ats['keyword_match_score']}%"
)

print(
    f"Experience Match  : "
    f"{actual_resume_ats['experience_match_score']}%"
)

print(
    f"Education Match   : "
    f"{actual_resume_ats['education_match_score']}%"
)


print("\n" + "-" * 70)
print("EXTRACTED RESUME SKILLS")
print("-" * 70)

for skill in actual_resume_ats["resume_skills"]:
    print("✅", skill)


print("\n" + "-" * 70)
print("JOB REQUIRED SKILLS")
print("-" * 70)

for skill in actual_resume_ats["required_skills"]:
    print("•", skill)


print("\n" + "-" * 70)
print("MATCHED SKILLS")
print("-" * 70)

for skill in actual_resume_ats["matched_skills"]:
    print("✅", skill)


print("\n" + "-" * 70)
print("MISSING SKILLS")
print("-" * 70)

for skill in actual_resume_ats["missing_skills"]:
    print("❌", skill)


print("\n" + "-" * 70)
print("EXPERIENCE")
print("-" * 70)

print(
    "Required:",
    actual_resume_ats["required_experience"],
    "years"
)

print(
    "Candidate:",
    actual_resume_ats["candidate_experience"],
    "years"
)


print("\n" + "-" * 70)
print("EDUCATION")
print("-" * 70)

print(
    "Required level:",
    actual_resume_ats["required_education_level"]
)

print(
    "Candidate level:",
    actual_resume_ats["candidate_education_level"]
)


print("\n" + "-" * 70)
print("SUGGESTIONS")
print("-" * 70)

for suggestion in actual_resume_ats["suggestions"]:
    print("💡", suggestion)


print("\n" + "=" * 70)
print("                  ANALYSIS COMPLETE")
print("=" * 70)

             YOUR RESUME - ATS ANALYSIS

FINAL ATS SCORE: 54.5%

----------------------------------------------------------------------
SCORE BREAKDOWN
----------------------------------------------------------------------
Skill Match       : 55.0%
Keyword Match     : 60.0%
Experience Match  : 0.0%
Education Match   : 100.0%

----------------------------------------------------------------------
EXTRACTED RESUME SKILLS
----------------------------------------------------------------------
✅ machine learning algorithms
✅ Computer Engineering
✅ Software Development
✅ Web Design
✅ REST APIs
✅ web-based
✅ project management
✅ role-based
✅ responsive templates
✅ DeepLearning.AI
✅ Node.js
✅ Express.js
✅ cloud deployment
✅ Scikit-learn
✅ REST API
✅ GitHub
✅ MongoDB
✅ preprocessing
✅ Python
✅ JavaScript
✅ Java
✅ C
✅ C++
✅ SQL
✅ HTML5
✅ CSS3
✅ Firebase
✅ Supabase
✅ NumPy
✅ Pandas
✅ Matplotlib
✅ Git
✅ Postman
✅ Vercel
✅ Netlify
✅ Render
✅ authentication
✅ database
✅ HTML
✅ CSS
✅ PDF
✅ React
✅ NG

In [55]:
print("=" * 70)
print("ALL EXTRACTED RESUME SKILLS")
print("=" * 70)

for i, skill in enumerate(actual_resume_ats["resume_skills"], 1):
    print(f"{i:02d}. {skill}")

print("\nTotal extracted:", len(actual_resume_ats["resume_skills"]))

ALL EXTRACTED RESUME SKILLS
01. machine learning algorithms
02. Computer Engineering
03. Software Development
04. Web Design
05. REST APIs
06. web-based
07. project management
08. role-based
09. responsive templates
10. DeepLearning.AI
11. Node.js
12. Express.js
13. cloud deployment
14. Scikit-learn
15. REST API
16. GitHub
17. MongoDB
18. preprocessing
19. Python
20. JavaScript
21. Java
22. C
23. C++
24. SQL
25. HTML5
26. CSS3
27. Firebase
28. Supabase
29. NumPy
30. Pandas
31. Matplotlib
32. Git
33. Postman
34. Vercel
35. Netlify
36. Render
37. authentication
38. database
39. HTML
40. CSS
41. PDF
42. React
43. NG

Total extracted: 43


In [56]:
import re

# ============================================================
# ATS SKILL NORMALIZATION + VALIDATION
# ============================================================

# Canonical skill names
SKILL_CANONICAL_MAP = {
    # Programming languages
    "python": "Python",
    "javascript": "JavaScript",
    "java": "Java",
    "c": "C",
    "c++": "C++",
    "c#": "C#",
    "sql": "SQL",

    # Frontend
    "react": "React",
    "react js": "React",
    "reactjs": "React",
    "react.js": "React",
    "html": "HTML",
    "html5": "HTML",
    "css": "CSS",
    "css3": "CSS",

    # Backend
    "node js": "Node.js",
    "nodejs": "Node.js",
    "node.js": "Node.js",
    "express js": "Express.js",
    "expressjs": "Express.js",
    "express.js": "Express.js",

    # APIs
    "rest api": "REST API",
    "rest apis": "REST API",
    "restful api": "REST API",
    "restful apis": "REST API",

    # Databases
    "mongodb": "MongoDB",
    "mongo db": "MongoDB",
    "mysql": "MySQL",
    "postgresql": "PostgreSQL",
    "postgres": "PostgreSQL",
    "firebase": "Firebase",
    "supabase": "Supabase",

    # ML / Data
    "machine learning": "Machine Learning",
    "machine-learning": "Machine Learning",
    "deep learning": "Deep Learning",
    "deep-learning": "Deep Learning",
    "numpy": "NumPy",
    "pandas": "Pandas",
    "matplotlib": "Matplotlib",
    "scikit learn": "Scikit-learn",
    "scikit-learn": "Scikit-learn",
    "sklearn": "Scikit-learn",

    # Tools
    "git": "Git",
    "github": "GitHub",
    "postman": "Postman",
    "docker": "Docker",
    "vs code": "VS Code",
    "visual studio code": "VS Code",

    # Cloud
    "aws": "AWS",
    "azure": "Azure",
    "google cloud": "Google Cloud",
    "google cloud run": "Google Cloud Run",
    "vercel": "Vercel",
    "netlify": "Netlify",
    "render": "Render",
}


# ============================================================
# PHRASES THAT SHOULD NOT BE TREATED AS SKILLS
# ============================================================

INVALID_SKILL_PHRASES = {
    "computer engineering",
    "software development",
    "web design",
    "web based",
    "web-based",
    "project management",
    "role based",
    "role-based",
    "responsive templates",
    "preprocessing",
    "machine learning algorithms",
    "deep learning algorithms",
    "cloud deployment",
    "software engineering",
    "engineering",
    "education",
    "experience",
    "knowledge",
    "development",
    "design",
    "management",
    "technology",
    "communication",
    "responsibility",
    "responsibilities",
    "work",
    "working",
    "ng",
}


# ============================================================
# ORGANIZATIONS / NON-SKILL ENTITIES
# ============================================================

NON_SKILL_ENTITIES = {
    "deeplearning.ai",
    "stanford university",
    "coursera",
    "udemy",
    "github",
    "linkedin",
    "leetcode",
}


# ============================================================
# NORMALIZE SKILL
# ============================================================

def normalize_ats_skill(skill):

    skill = str(skill).strip()

    if not skill:
        return None

    # Basic normalization
    normalized = skill.lower().strip()

    normalized = normalized.replace("–", "-")
    normalized = normalized.replace("—", "-")

    # Remove surrounding punctuation
    normalized = re.sub(r"^[^\w+#]+|[^\w+#.]+$", "", normalized)

    # Normalize spaces
    normalized = re.sub(r"\s+", " ", normalized).strip()

    # Reject obvious invalid phrases
    if normalized in INVALID_SKILL_PHRASES:
        return None

    # Reject organizations / non-skill entities
    if normalized in NON_SKILL_ENTITIES:
        return None

    # Canonical mapping
    if normalized in SKILL_CANONICAL_MAP:
        return SKILL_CANONICAL_MAP[normalized]

    # Remove obvious garbage
    if len(normalized) <= 2:
        return None

    return skill.strip()


# ============================================================
# REMOVE DUPLICATES
# ============================================================

def clean_ats_skill_list(skills):

    cleaned = []

    seen = set()

    for skill in skills:

        normalized_skill = normalize_ats_skill(skill)

        if normalized_skill is None:
            continue

        key = normalize_skill_text(normalized_skill)

        if key in seen:
            continue

        seen.add(key)
        cleaned.append(normalized_skill)

    return cleaned


# ============================================================
# FINAL ATS SKILL EXTRACTOR
# ============================================================

def ats_extract_skills(resume_text):

    # Existing hybrid extractor
    raw_skills = hybrid_extract_skills(resume_text)

    # Clean and normalize
    final_skills = clean_ats_skill_list(raw_skills)

    return final_skills


print("ATS skill cleaning system ready.")

ATS skill cleaning system ready.


In [57]:
cleaned_resume_skills = ats_extract_skills(sample_resume)

print("=" * 70)
print("CLEANED ATS SKILLS")
print("=" * 70)

for i, skill in enumerate(cleaned_resume_skills, 1):
    print(f"{i:02d}. {skill}")

print("\nTotal:", len(cleaned_resume_skills))

CLEANED ATS SKILLS
01. REST API
02. Node.js
03. Express.js
04. Scikit-learn
05. MongoDB
06. Python
07. JavaScript
08. Java
09. C
10. C++
11. SQL
12. HTML
13. CSS
14. Firebase
15. Supabase
16. NumPy
17. Pandas
18. Matplotlib
19. Git
20. Postman
21. Vercel
22. Netlify
23. Render
24. authentication
25. database
26. PDF
27. React

Total: 27


In [61]:
# ============================================================
# FINAL ATS ANALYZER — CLEANED VERSION
# ============================================================

def final_ats_analyzer(
    resume_text,
    job_description,
    candidate_experience=None,
    candidate_education=None
):

    # --------------------------------------------------------
    # 1. CLEAN SKILL EXTRACTION
    # --------------------------------------------------------

    resume_skills = ats_extract_skills(resume_text)
    required_skills = ats_extract_skills(job_description)

    # --------------------------------------------------------
    # 2. NORMALIZE SKILLS FOR MATCHING
    # --------------------------------------------------------

    resume_map = {}

    for skill in resume_skills:
        key = normalize_skill_text(skill)
        resume_map[key] = skill

    required_map = {}

    for skill in required_skills:
        key = normalize_skill_text(skill)
        required_map[key] = skill

    # --------------------------------------------------------
    # 3. SKILL MATCHING
    # --------------------------------------------------------

    matched_skills = []
    missing_skills = []

    for key, required_skill in required_map.items():

        if key in resume_map:
            matched_skills.append(required_skill)
        else:
            missing_skills.append(required_skill)

    if len(required_map) > 0:
        skill_score = (
            len(matched_skills) /
            len(required_map)
        ) * 100
    else:
        skill_score = 0

    skill_score = round(skill_score, 2)

    # --------------------------------------------------------
    # 4. KEYWORD MATCHING
    # --------------------------------------------------------

    resume_normalized = normalize_skill_text(
        resume_text
    )

    matched_keywords = []
    missing_keywords = []

    for skill in required_skills:

        skill_normalized = normalize_skill_text(
            skill
        )

        if skill_normalized in resume_normalized:
            matched_keywords.append(skill)
        else:
            missing_keywords.append(skill)

    if len(required_skills) > 0:

        keyword_score = (
            len(matched_keywords) /
            len(required_skills)
        ) * 100

    else:
        keyword_score = 0

    keyword_score = round(
        keyword_score,
        2
    )

    # --------------------------------------------------------
    # 5. EXPERIENCE MATCH
    # --------------------------------------------------------

    experience_result = calculate_experience_match(
        candidate_experience,
        job_description
    )

    experience_score = experience_result[
        "experience_score"
    ]

    # --------------------------------------------------------
    # 6. EDUCATION MATCH
    # --------------------------------------------------------

    education_result = calculate_education_match(
        candidate_education,
        job_description
    )

    education_score = education_result[
        "education_score"
    ]

    # --------------------------------------------------------
    # 7. FINAL WEIGHTED ATS SCORE
    # --------------------------------------------------------

    final_score = (

        skill_score * 0.50

        + keyword_score * 0.20

        + experience_score * 0.15

        + education_score * 0.15
    )

    final_score = round(
        final_score,
        2
    )

    # --------------------------------------------------------
    # 8. SUGGESTIONS
    # --------------------------------------------------------

    suggestions = []

    for skill in missing_skills:

        suggestions.append(
            f"Consider adding '{skill}' "
            f"if you have relevant experience."
        )

    required_exp = experience_result[
        "required_experience"
    ]

    if required_exp is not None:

        if candidate_experience is None:

            suggestions.append(
                "Add your relevant years of "
                "experience to the resume."
            )

        elif candidate_experience < required_exp:

            suggestions.append(
                f"The job requires approximately "
                f"{required_exp}+ years of experience."
            )

    if final_score >= 80:

        suggestions.append(
            "The resume has strong alignment "
            "with the analyzed job description."
        )

    elif final_score >= 60:

        suggestions.append(
            "The resume has moderate alignment. "
            "Review the missing skills and requirements."
        )

    else:

        suggestions.append(
            "The resume has limited alignment. "
            "Review the required skills, experience "
            "and education."
        )

    # --------------------------------------------------------
    # 9. RETURN COMPLETE RESULT
    # --------------------------------------------------------

    return {

        "final_ats_score": final_score,

        "skill_match_score": skill_score,

        "keyword_match_score": keyword_score,

        "experience_match_score": experience_score,

        "education_match_score": education_score,

        "resume_skills": resume_skills,

        "required_skills": required_skills,

        "matched_skills": matched_skills,

        "missing_skills": missing_skills,

        "matched_keywords": matched_keywords,

        "missing_keywords": missing_keywords,

        "required_experience":
            experience_result[
                "required_experience"
            ],

        "candidate_experience":
            experience_result[
                "candidate_experience"
            ],

        "required_education_level":
            education_result[
                "required_education_level"
            ],

        "candidate_education_level":
            education_result[
                "candidate_education_level"
            ],

        "suggestions": suggestions
    }


# ============================================================
# RUN ON YOUR RESUME
# ============================================================

actual_resume_ats = final_ats_analyzer(

    resume_text=sample_resume,

    job_description=sample_job_description,

    # Your resume does not state professional experience.
    # Projects are not automatically counted as years.
    candidate_experience=0,

    # B.Tech = level 3 in our current system.
    candidate_education=3
)


# ============================================================
# COMPLETE ATS REPORT
# ============================================================

print("=" * 75)
print("                 FINAL ATS ANALYSIS REPORT")
print("=" * 75)

print(
    f"\nFINAL ATS SCORE: "
    f"{actual_resume_ats['final_ats_score']}%"
)


print("\n" + "-" * 75)
print("SCORE BREAKDOWN")
print("-" * 75)

print(
    f"Skill Match       : "
    f"{actual_resume_ats['skill_match_score']}%  "
    f"(Weight: 50%)"
)

print(
    f"Keyword Match     : "
    f"{actual_resume_ats['keyword_match_score']}%  "
    f"(Weight: 20%)"
)

print(
    f"Experience Match  : "
    f"{actual_resume_ats['experience_match_score']}%  "
    f"(Weight: 15%)"
)

print(
    f"Education Match   : "
    f"{actual_resume_ats['education_match_score']}%  "
    f"(Weight: 15%)"
)


print("\n" + "-" * 75)
print("EXTRACTED RESUME SKILLS")
print("-" * 75)

for i, skill in enumerate(
    actual_resume_ats["resume_skills"],
    1
):
    print(f"{i:02d}. {skill}")

print(
    f"\nTotal Resume Skills: "
    f"{len(actual_resume_ats['resume_skills'])}"
)


print("\n" + "-" * 75)
print("REQUIRED JOB SKILLS")
print("-" * 75)

for i, skill in enumerate(
    actual_resume_ats["required_skills"],
    1
):
    print(f"{i:02d}. {skill}")

print(
    f"\nTotal Required Skills: "
    f"{len(actual_resume_ats['required_skills'])}"
)


print("\n" + "-" * 75)
print("MATCHED SKILLS")
print("-" * 75)

if actual_resume_ats["matched_skills"]:

    for skill in actual_resume_ats["matched_skills"]:
        print("✅", skill)

else:

    print("None")


print("\n" + "-" * 75)
print("MISSING SKILLS")
print("-" * 75)

if actual_resume_ats["missing_skills"]:

    for skill in actual_resume_ats["missing_skills"]:
        print("❌", skill)

else:

    print("None")


print("\n" + "-" * 75)
print("EXPERIENCE ANALYSIS")
print("-" * 75)

print(
    "Required Experience:",
    actual_resume_ats["required_experience"],
    "years"
)

print(
    "Candidate Experience:",
    actual_resume_ats["candidate_experience"],
    "years"
)

print(
    "Experience Score:",
    actual_resume_ats["experience_match_score"],
    "%"
)


print("\n" + "-" * 75)
print("EDUCATION ANALYSIS")
print("-" * 75)

print(
    "Required Education Level:",
    actual_resume_ats["required_education_level"]
)

print(
    "Candidate Education Level:",
    actual_resume_ats["candidate_education_level"]
)

print(
    "Education Score:",
    actual_resume_ats["education_match_score"],
    "%"
)


print("\n" + "-" * 75)
print("ATS SUGGESTIONS")
print("-" * 75)

for suggestion in actual_resume_ats["suggestions"]:

    print("💡", suggestion)


print("\n" + "=" * 75)
print("                    ANALYSIS COMPLETE")
print("=" * 75)

                 FINAL ATS ANALYSIS REPORT

FINAL ATS SCORE: 53.23%

---------------------------------------------------------------------------
SCORE BREAKDOWN
---------------------------------------------------------------------------
Skill Match       : 52.94%  (Weight: 50%)
Keyword Match     : 58.82%  (Weight: 20%)
Experience Match  : 0.0%  (Weight: 15%)
Education Match   : 100.0%  (Weight: 15%)

---------------------------------------------------------------------------
EXTRACTED RESUME SKILLS
---------------------------------------------------------------------------
01. REST API
02. Node.js
03. Express.js
04. Scikit-learn
05. MongoDB
06. Python
07. JavaScript
08. Java
09. C
10. C++
11. SQL
12. HTML
13. CSS
14. Firebase
15. Supabase
16. NumPy
17. Pandas
18. Matplotlib
19. Git
20. Postman
21. Vercel
22. Netlify
23. Render
24. authentication
25. database
26. React

Total Resume Skills: 26

---------------------------------------------------------------------------
REQUIRED JOB SKIL

In [63]:
INVALID_SKILL_PHRASES.update({
    "computer science",
    "information technology",
    "computer engineering",
    "information systems",
    "information science",
    "electrical engineering",
    "mechanical engineering",
    "electronics engineering",
    "civil engineering",
    "bachelor",
    "bachelors",
    "bachelor degree",
    "bachelor's degree",
    "master",
    "masters",
    "master degree",
    "master's degree",
    "undergraduate",
    "postgraduate",
    "degree",
})